### **Job Script Workflows 1**


We wrote this:
```bash
#!/bin/bash
#BSUB -J myarray[1-10]
#BSUB -q hpc
#BSUB -n 1
#BSUB -R "span[hosts=1]"
#BSUB -W 00:05
#BSUB -o myarray_%J_%I.out
#BSUB -e myarray_%J_%I.err

source /dtu/projects/02613_2025/conda/conda_init.sh

conda activate 02613

python preprocess.py input001.png
```

### **Job Script Workflows 2**


We just made:
```bash
#!/bin/bash
#BSUB -J myindices[2,29,71,73,127]
#BSUB -q hpc
#BSUB -n 1
#BSUB -R "span[hosts=1]"
#BSUB -W 00:05
#BSUB -o myindices_%J_%I.out
#BSUB -e myindices_%J_%I.err

source /dtu/projects/02613_2025/conda/conda_init.sh

conda activate 02613

python preprocess.py input001.png
```

### **Job Script Workflows 3**


We just wrote:
```bash
#!/bin/bash
#BSUB -J job3
#BSUB -q hpc
#BSUB -n 1
#BSUB -R "span[hosts=1]"
#BSUB -W 00:05
#BSUB -w 1234567
#BSUB -o job3_%J.out
#BSUB -e job3_%J.err

source /dtu/projects/02613_2025/conda/conda_init.sh

conda activate 02613

python preprocess.py input001.png
```

### **Job Script Workflows 4**


We have to assume the following output after running `bstat` on a job array:
```bash
$ bstat
JOBID      USER    QUEUE      JOB_NAME   NALLOC STAT  START_TIME      ELAPSED
21241475   patmjen hpc        array[1]        1 RUN   Apr  9 13:40    0:00:09
21241475   patmjen hpc        array[2]        1 RUN   Apr  9 13:40    0:00:09
21241475   patmjen hpc        array[3]        1 RUN   Apr  9 13:40    0:00:09
21241475   patmjen hpc        array[4]        1 RUN   Apr  9 13:40    0:00:09
21241475   patmjen hpc        array[5]        1 RUN   Apr  9 13:40    0:00:09
```

We have to make job script that creates a job array, where each job in the array waits for the corresponding job in the already submitted array to finish successfully.
```bash
#!/bin/bash
#BSUB -J myindices[1-5]
#BSUB -q hpc
#BSUB -n 1
#BSUB -R "span[hosts=1]"
#BSUB -W 00:05
#BSUB -w done(21241475[*])
#BSUB -o myindices_%J_%I.out
#BSUB -e myindices_%J_%I.err

source /dtu/projects/02613_2025/conda/conda_init.sh

conda activate 02613

python preprocess.py input001.png
```

### **Job Script Workflows 5**


We write:
```bash
#!/bin/bash
#BSUB -J myindices[1-5]
#BSUB -q hpc
#BSUB -n 1
#BSUB -R "span[hosts=1]"
#BSUB -W 00:05
#BSUB -w ended(21241475)
#BSUB -o myindices_%J_%I.out
#BSUB -e myindices_%J_%I.err

source /dtu/projects/02613_2025/conda/conda_init.sh

conda activate 02613

python preprocess.py input001.png
```

### **Face Colors 1**

We have to consider the following function:
```python
import numpy as np
from PIL import Image

def huehist(image):
    bins = np.linspace(0, 255, 64 + 1)
    hsv_image = np.asarray(Image.fromarray(image).convert('HSV'))
    hue_values = hsv_image[:, :, 0].reshape(-1)
    hue_hist = np.histogram(hue_values, bins)[0]
    return hue_hist
```
We write the following code:
```python
import sys
import os
import numpy as np

from PIL import Image

def huehist(image):
    bins = np.linspace(0, 255, 64 + 1)
    hsv_image = np.asarray(Image.fromarray(image).convert('HSV'))
    hue_values = hsv_image[:, :, 0].reshape(-1)
    hue_hist = np.histogram(hue_values, bins)[0]
    return hue_hist

base_path = "/dtu/projects/02613_2024/data/celeba/images"

i = int(sys.argv[1])

subfolders = sorted(os.listdir(base_path))
folder = subfolders[i]
folder_path = os.path.join(base_path, folder)

image_files = glob.glob(os.path.join(folder_path, "*"))
images = [np.array(Image.open(img)) for img in image_files]

hue_hists = [huehist(img) for img in images]
hist_sum_folder = np.sum(hue_hists, axis=0)

np.save(os.path.join("hist_sums", f"subhist_{i}.npy"), hist_sum_folder)
```

### **Face Colors 2**

Now we have to create a Python program that loads the saved histogram arrays, sums them to a combined histogram for the entire dataset, plots the histogram using plt.bar and finally saves the histogram plot as an image. Here:
```python
def plot_hists(path):
    hist_files = glob.glob(os.path.join(path, "subhist_*.npy"))
    histograms = [np.load(hist_file) for hist_file in hist_files]
    combined_histogram = np.sum(histograms, axis=0)
    plt.bar(range(len(combined_histogram)), combined_histogram)
    plt.xlabel("Hue bin")
    plt.ylabel("Frequency")
    plt.title("Combined hue histogram")
    plt.savefig(os.path.join(path, "combined_histogram.png"))
```

### **Face Colors 3**

Now we have to make a job array thingie:
```bash
#!/bin/bash
#BSUB -J myindices[0-"number of subfolders"]
#BSUB -q hpc
#BSUB -n 1
#BSUB -R "span[hosts=1]"
#BSUB -W 00:05
#BSUB -o myindices_%J_%I.out
#BSUB -e myindices_%J_%I.err

source /dtu/projects/02613_2025/conda/conda_init.sh

conda activate 02613

python face_colors.py $LSB_JOBINDEX
```

We check number of subfolders using `ls /dtu/projects/02613_2024/data/celeba/images | wc -l` but I am too lazy to connect to VPN qwq.

THEN a job scripts for aggregation:
```bash
#!/bin/bash
#BSUB -J celeba_final
#BSUB -q hpc
#BSUB -n 1
#BSUB -R "span[hosts=1]"
#BSUB -W 00:05
#BSUB -w done(celeba)
#BSUB -o celeba_final_%J.out
#BSUB -e celeba_final_%J.err

source /dtu/projects/02613_2025/conda/conda_init.sh
conda activate 02613

python aggregate.py
```

### **Face Colors 4**

We do:
```bash
#!/bin/bash
#BSUB -J celeba_final
#BSUB -q hpc
#BSUB -n 1
#BSUB -R "span[hosts=1]"
#BSUB -W 00:05
#BSUB -w done(celeba)
#BSUB -o celeba_final_%J.out
#BSUB -e celeba_final_%J.err

source /dtu/projects/02613_2025/conda/conda_init.sh
conda activate 02613

python aggregate.py
```

### **Face Colors 5**

To lazy to do this qwq oops.